In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

from pathlib import Path
from tqdm import tqdm
import pickle

# Generate hotspots of unfairness based on movement patterns

#### Read the Atlanta's 2020 US Census blocks and the stop segments

In [ ]:
path_atlanta_blocks = './experiments/atlanta_census_blocks.zip'
atlanta_blocks = gpd.read_file(path_atlanta_blocks)['geometry'].to_crs("EPSG:4326").to_frame()
atlanta_blocks


path_stop_df = './data_simulator/huge_dataset/dataset_simulator_trajectories.compressed.parquet.stops.parquet'
stop_df = pd.read_parquet(path_stop_df)
stop_df = gpd.GeoDataFrame(stop_df, 
                           geometry=gpd.points_from_xy(stop_df.lng, stop_df.lat), 
                           crs="EPSG:4326").loc[:, ['uid', 'geometry']]
display(stop_df)

#### Estimate a metric CRS we can use for some manipulations for objects that fall over the area covered by the census blocks.

In [ ]:
orig_crs = atlanta_blocks.crs
metric_crs = atlanta_blocks.estimate_utm_crs()
# print(orig_crs, metric_crs)


# Reproject both blocks and stop segments' centroids.
atlanta_blocks = atlanta_blocks.to_crs(metric_crs)
stop_df = stop_df.to_crs(metric_crs)

#### Filter out the blocks that do not contain any stop segment.

In [ ]:
# Associated each stop segment to a census block via its centroid.
# This will be useful when building hotspots made of multiple separate regions: we can use the candidate
# generation algorithm to see where there are objects associated with more than 1 separate region, and use them
# as seeds to build this kind of hotspots.
mapped_stops = stop_df.sjoin(atlanta_blocks,
                             how="left",
                             predicate="within")[['uid', 'index_right']]

# Retrieve the rtree that has been built over the points right before the spatial join -- we will use it
# to perform queries with arbitrary polyglns over the stops' centroids.
stop_df_rtree = stop_df.sindex


# Filter out the blocks that do not contain any stop segment.
list_nonempty_blocks = mapped_stops['index_right'].unique()
# display(list_nonempty_blocks)

sel_atlanta_blocks = atlanta_blocks.loc[list_nonempty_blocks].copy()
# sel_atlanta_blocks.plot()

# Remove 'mapped_stops' from memory (no more necessary for here on).
del mapped_stops

#### Generate the hotspots of unfairness based on movement patterns

In [ ]:
from src.gen_unfair_hotspots import gen_hotspot, gen_set_hotspots
 
hotspot = gen_set_hotspots(sel_atlanta_blocks, stop_df, 1000, 600)

In [ ]:
list_num_objs = [h[1].size for h in hotspot]
np.mean(list_num_objs), np.std(list_num_objs)

### Write the synthetic unfair labels to disk

### DEBUG: Plot the original shape of a polygon, and its shrunken+rotated+translated version.

**DEBUG**: plot a simple Folium map of the Atlanta's tracts -- nonempty vs empty.